# Towards Universal Storm Sampling Method
In this notebook I will try to develop the whole universal storm sampling method with Non-Homogeneous Poisson Process. 

The step in generating storm sample are the following: 
1. Import wave time-series data 
2. Detect the wave storm based on the wave time-series 
3. Fitting the storm into statistical distribution 
4. Build synthetic wave storm time series 

In [ ]:
import matplotlib.pyplot as plt
import numpy as np 
import pandas as pd
import xarray as xr

from datetime import datetime, timedelta
from pcr import helper, storm

# libraries for interactive plots
import hvplot.pandas  # noqa
import hvplot.xarray  # noqa
import holoviews as hv

from holoviews import opts

In [ ]:
# access from the pre-downloaded 
data_path = '../data/ERA5/ts/sri_lanka_1979_2020.nc'
ds = xr.open_dataset(data_path)

print(f'longitude: {ds.longitude.values}')
print(f'latitude: {ds.latitude.values}')


In [ ]:
ds['swh'].hvplot.line(
    grid=True, 
    title='Significant Wave Height Time Series at Sri Lanka (81.50E, 8.50N)',
    xlabel='Time',
    ylabel='Significant Wave Height (m)'
)

In [ ]:
# detect storm 
ts_hs = 95 
ts_dur = 12 

hs, dir, tp, time = helper.era5_input(ds)

detected_storm, _ = storm.detect(hs, dir, tp, time, ts_hs, ts_dur, ts_between=48)

# fit storm and gap 
fitted_storms = storm.fit_storm(detected_storm)
lambda_storm = storm.fit_lambda(detected_storm, fillna='zeros')

n_sampling = detected_storm.shape[0]

# generate storm sample
hs, dur, dir, tp = storm.generate(
    fitted_storm=fitted_storms, 
    sampling_size=n_sampling, 
    oversample=0.25, 
    max_dur=np.max(detected_storm.duration))

storms_sample = pd.DataFrame(
        {
            "hs": hs,
            "duration": dur,
            "direction": dir,
            "tp": tp
        }, 
        copy=False
    )

In [ ]:
# Define the time horizon (same as the ERA5 data period)
date_start = np.datetime64('1979-01-01')
date_end = np.datetime64('2020-12-31T23:00:00')

t_days = (date_end-date_start).item().days # time horizon in days

storm_start = storm.simulate_nhpp_thinning(t_days, lambda_storm, date_start) # in days since date_start

## Experiment: Accept-reject inter-arrival time
accept the arrival if the inter-arrival time is larger than the storm duration

In [ ]:
# try another way to simulate the storm start times

lambda_max = lambda_storm.max() + 1
t = 0
arrivals = []
T = t_days
counter = 0
other_counter = 0

while True:
    # Step 1: propose next arrival in Poisson(λ_max)
    # Gap ~ Exponential(λ_max)
    gap = np.random.exponential(1.0 / lambda_max) * 365.25  # convert to days
    t = t + gap
    if t > T:
        break

    # Step 2: accept with probability λ(t) / λ_max
    u = np.random.uniform(0.0, 1.0)
    if u <= storm.get_lambda(t, lambda_storm, date_start) / lambda_max:
        # accept if the gap is larger than the storm duration 
        if gap >= (dur[counter] / 24):
            arrivals.append(t)
            counter += 1
        else:
            # reject, go back one step
            t = t - gap
            other_counter += 1
    

**Result:**
the distribution of storm count is way to low 

## Experiment: jump after the storm duration
after sampling the inter-arrival time, advance time with the storm duration

In [ ]:
def modified_nhpp_thinning(t_days, lambda_storm, date_start, duration):
    """
    Return to arrival (start) of an event after a the end of the event (gap)
    
    :param T: time horizon in days 
    :param monthly_lambda: array of event intensity for each month (size of 12)
    :param date_start: datetime of the start of the simulation
    :param duration: an array-like of duration
    :return: array of start of each storm 
    """
    # try another way to simulate the storm start times
    lambda_max = lambda_storm.max()
    t = 0
    arrivals = []
    T = t_days
    counter = 0

    while True:
        # Step 1: propose next arrival in Poisson(λ_max)
        # Gap ~ Exponential(λ_max)
        gap = np.random.exponential(1.0 / lambda_max) * 365.25  # convert to days
        t += gap
        if t > T:
            break

        # Step 2: accept with probability λ(t) / λ_max
        u = np.random.uniform(0.0, 1.0)
        if u <= storm.get_lambda(t, lambda_storm, date_start) / lambda_max:
            arrivals.append(t)
            t += duration[counter] / 24
            counter += 1
    
    return np.array(arrivals)

In [ ]:
arrivals_mod = modified_nhpp_thinning(t_days, lambda_storm, date_start, dur)

print(f'Number of storm from data: {detected_storm.shape[0]}')
print(f'Number of simulated storms with NHPP: {len(storm_start)}')
# print(f'Number of simulated storms with modified NHPP: {counter}')
print(f'Number of simulated storms with modified NHPP: {len(arrivals_mod)}')


In [ ]:
numsim = 10000

res1 = []
for i in range(numsim) :
    res1.append(modified_nhpp_thinning(t_days, lambda_storm, date_start, dur))

In [ ]:
# Make a histogram of number of storms in each simulation
import matplotlib.pyplot as plt

N = [res.size for res in res1]

plt.figure(figsize=(8,5))
plt.hist(N, bins=50, alpha=0.7, color='tab:blue', edgecolor='black', label='NHPP Thinning Simulation')
plt.axvline(detected_storm.shape[0], color='red', linestyle='dashed', linewidth=2, label='Observed Data')
plt.xlabel("Number of Storms in 41 years")
plt.ylabel("Frequency")
plt.title("Histogram of Number of Storms in 10,000 NHPP Thinning Simulations")
plt.legend()
plt.show()

print(f'Mean: {np.mean(N)}')
print(f'Median: {np.median(N)}')
print(f'Data: {detected_storm.shape[0]}')

In [ ]:
# get the average number of storms per month from the simulations
count_sim = pd.DataFrame(
    {'count': np.zeros(12)},
    index=np.arange(1,13),
)

for i in range(numsim):
    out_sl = pd.DataFrame({
        'time':[date_start + np.timedelta64(int(start*24), 'h') for start in res1[i]], 
    })

    out_sl['month'] = [s_dt.month for s_dt in out_sl['time']]

    count_sim['addition'] = out_sl.groupby('month').count()['time']
    count_sim['addition'] = count_sim['addition'].fillna(0)
    count_sim['count'] = count_sim['count'] + count_sim['addition']

count_sim['avg_count'] = count_sim['count'] / numsim

# Plot the monthly counts
plt.figure(figsize=(10, 5))
width = 0.4
x = np.arange(1, 13)

plt.grid(linestyle='--', alpha=0.3, zorder=0)
plt.bar(x-width/2, count_sim['avg_count'], width, label= "10,000 Simulation")
plt.bar(x+width/2, (lambda_storm * 41 / 12), width, label= "Data")
plt.xticks(x, ['Jan', 'Feb', 'Mar', 'Apr', 'May', 'Jun', 'Jul', 'Aug', 'Sep', 'Oct', 'Nov', 'Dec'])
plt.xlabel("Month")
plt.ylabel("Number of Storms")
plt.title("Monthly Storm Counts: 10,000 HNPP Thinning Simulation vs Data")
plt.legend()

plt.show()

**Results:** Still to low  
from the graph we see that we have significantly lower monthly storm count in December and January. It is because that they cannot fit enough storm within months because we fit the lambda with the **inter-arrival time** of storms (time from storm start to the subsequent start).  

Ideally, the rate of inter-arrival time should be fitted with the **gap** between storm (time from storm ends to the subsequent start of the storm)

## Find intensity of event based on gaps 
The idea of this is to have the described NHPP with thinning method to sample gaps appropriately. To sample gaps, we need an *increased* rate, because it has smaller duration relative to the inter-arrival time.  
The function is implemented in `storm` module as `fit_lambda_gap` function 

In [ ]:
lambdas = storm.fit_lambda_gap(detected_storm, 'zeros')

In [ ]:
# note: it is quite slow -> 
# due to higher lambda in the modified one -> 
# drawing more candidate time (while more candidate rejected)
numsim = 10000

res2 = []
for i in range(numsim) :
    res2.append(modified_nhpp_thinning(t_days, lambdas, date_start, dur))

In [ ]:
N = [res.size for res in res2]

plt.figure(figsize=(8,5))
plt.hist(N, bins=50, alpha=0.7, color='tab:blue', edgecolor='black', label='NHPP Thinning Simulation')
plt.axvline(detected_storm.shape[0], color='red', linestyle='dashed', linewidth=2, label='Observed Data')
plt.xlabel("Number of Storms in 41 years")
plt.ylabel("Frequency")
plt.title("Histogram of Number of Storms in 10,000 NHPP Thinning Simulations")
plt.legend()
plt.show()

In [ ]:
print(f'median: {np.median(N)}')
print(f'mean: {np.mean(N)}')
print(f'data: {detected_storm.shape[0]}')

In [ ]:
# get the average number of storms per month from the simulations
count_sim = pd.DataFrame(
    {'count': np.zeros(12)},
    index=np.arange(1,13),
)

for i in range(numsim):
    out_sl = pd.DataFrame({
        'time':[date_start + np.timedelta64(int(start*24), 'h') for start in res2[i]], 
    })

    out_sl['month'] = [s_dt.month for s_dt in out_sl['time']]

    count_sim['addition'] = out_sl.groupby('month').count()['time']
    count_sim['addition'] = count_sim['addition'].fillna(0)
    count_sim['count'] = count_sim['count'] + count_sim['addition']

count_sim['avg_count'] = count_sim['count'] / numsim

# Plot the monthly counts
plt.figure(figsize=(10, 5))
width = 0.4
x = np.arange(1, 13)

plt.grid(linestyle='--', alpha=0.3, zorder=0)
plt.bar(x-width/2, count_sim['avg_count'], width, label= "10,000 Simulation")
plt.bar(x+width/2, (lambda_storm * 41 / 12), width, label= "Data")
plt.xticks(x, ['Jan', 'Feb', 'Mar', 'Apr', 'May', 'Jun', 'Jul', 'Aug', 'Sep', 'Oct', 'Nov', 'Dec'])
plt.xlabel("Month")
plt.ylabel("Number of Storms")
plt.title("Monthly Storm Counts: 10,000 HNPP Thinning Simulation vs Data")
plt.legend()

plt.show()

In [ ]:
# CDF of gaps 
gaps = []
durs = storms_sample['duration'].values

for i in range(10000):
    # start in day 
    one_sim = res2[i]
    # duration in hour
    dur = durs[:len(one_sim)]

    # end of storm (in days)
    end_storm = one_sim + dur / 24

    # gap: start(i) - end(i-1) (in days)
    gap = one_sim[1:] - end_storm[:-1]
    gaps.append(gap)

all_gaps = np.concatenate(gaps)
gap_data = detected_storm['gap'][1:].values

gap_data.sort()
all_gaps.sort()

plt.figure(figsize=(10,6))
plt.plot(gap_data,np.arange(len(gap_data))/len(gap_data), label='Data')
plt.plot(all_gaps,np.arange(len(all_gaps))/len(all_gaps), '--', label='Simulated')
plt.grid(linestyle='--', alpha=0.3, zorder=0)
plt.title('Empirical CDF for Gap, Data vs 100,000 Simulation')
plt.xlabel('Gaps (days)')
plt.legend()
plt.ylabel('CDF')
plt.show()


**Results:** Fitting the lambda to the gap and using the thinning to draw the gap does a good job 

# Implement it to other location
implement the functions to location P2, P11, and P24 
## Define the Function  

In [ ]:
def go_simulate(ds):
    # detect storm 
    ts_hs = 95 
    ts_dur = 12 

    hs, dir, tp, time = helper.era5_input(ds)

    detected_storm, _ = storm.detect(hs, dir, tp, time, ts_hs, ts_dur, ts_between=48)

    # fit storm and gap 
    fitted_storms = storm.fit_storm(detected_storm)
    lambda_storm = storm.fit_lambda_gap(detected_storm, fillna='zeros')

    n_sampling = detected_storm.shape[0]

    # generate hs, dur, tp, direction sample
    _, durs, _, _ = storm.generate(
        fitted_storm=fitted_storms, 
        sampling_size=n_sampling, 
        oversample=0.25, 
        max_dur=np.max(detected_storm.duration))

    # Define the time horizon (same as the data period)
    date_start = ds.valid_time[0].values.astype('datetime64[s]')
    date_end = ds.valid_time[-1].values.astype('datetime64[s]')

    t_days = (date_end-date_start).item().days # time horizon in days

    storm_start = modified_nhpp_thinning(t_days, lambda_storm, date_start, durs) # in days since date_start

    return storm_start, durs[:len(storm_start)], detected_storm

def go_simulate_multiple(ds, n_sims):
    # detect storm 
    ts_hs = 95 
    ts_dur = 12 

    hs, dir, tp, time = helper.era5_input(ds)

    detected_storm, _ = storm.detect(hs, dir, tp, time, ts_hs, ts_dur, ts_between=48)

    # fit storm and gap 
    fitted_storms = storm.fit_storm(detected_storm)
    lambda_storm = storm.fit_lambda_gap(detected_storm, fillna='zeros')

    # generate hs, dur, tp, direction sample
    n_sampling = detected_storm.shape[0] * n_sims

    _, durs, _, _ = storm.generate(
        fitted_storm=fitted_storms, 
        sampling_size=n_sampling, 
        oversample=0.1, 
        max_dur=np.max(detected_storm.duration))

    # Define the time horizon (same as the data period)
    date_start = ds.valid_time[0].values.astype('datetime64[s]')
    date_end = ds.valid_time[-1].values.astype('datetime64[s]')

    t_days = (date_end-date_start).item().days # time horizon in days
    
    # durs = storms_sample['duration'].values
    res = []
    durs_sim = []
    # sample_idx = []
    counter = 0

    for i in range(n_sims):
        dur = durs[counter:]
        res.append(modified_nhpp_thinning(t_days, lambda_storm, date_start, dur)) # in days since date_start
        
        storm_count = len(res[i])
        durs_sim.append(dur[:storm_count])
        
        counter += storm_count
        
    return res, durs_sim, detected_storm

In [ ]:
# bar chart per month 
def compare_bar_month(dt_1:np.array, dt_2:np.array, label_1: str='Data 1', label_2: str='Data 2', title:str=''):
    
    
    df_1 = pd.DataFrame({'start': dt_1})
    df_2 = pd.DataFrame({'start': dt_2})

    count_sim = pd.DataFrame(
        index=np.arange(1,13),
    )

    count_sim[label_1] = df_1.groupby(by=df_1['start'].dt.month).count()
    count_sim[label_2] = df_2.groupby(by=df_2['start'].dt.month).count()

    # Plot
    fig, ax = plt.subplots(figsize=(10,5))
    width = 0.4
    x = np.arange(1, 13)

    ax.grid(linestyle='--', alpha=0.3, zorder=0)
    ax.bar(x - width/2, count_sim[label_1], width, label="Simulation")
    ax.bar(x + width/2, count_sim[label_2], width, label="Data")

    ax.set_xticks(x)
    ax.set_xticklabels(['Jan', 'Feb', 'Mar', 'Apr', 'May', 'Jun',
                         'Jul', 'Aug', 'Sep', 'Oct', 'Nov', 'Dec'])
    ax.set_xlabel("Month")
    ax.set_ylabel("Number of Storms")
    ax.set_title(title)
    ax.legend()

    plt.show()

    return fig, ax

# compare empirical CDF 
def get_gap(storm_start, durs):
    
    end_storm = storm_start + (durs / 24)
    gap = storm_start[1:] - end_storm[:-1]
    
    return gap 

def compare_ecdf(storm_start, durs, detected_storm_):

    # CDF of gaps
    if isinstance(storm_start, list):
        gap_sim = np.concatenate([get_gap(res, dur_pairs) for res, dur_pairs in zip(storm_start, durs)])
    else: 
        gap_sim = get_gap(storm_start, durs)

    gap_data = detected_storm_['gap'][1:].values

    # Sort for CDF
    gap_sim = np.sort(gap_sim)
    gap_data = np.sort(gap_data)

    # Empirical CDFs
    cdf_sim = np.arange(len(gap_sim)) / len(gap_sim)
    cdf_data = np.arange(len(gap_data)) / len(gap_data)

    # Plot
    fig, ax = plt.subplots(figsize=(10,6))

    ax.plot(gap_data, cdf_data, label='Data')
    ax.plot(gap_sim, cdf_sim, '--', label='Simulated')

    ax.grid(linestyle='--', alpha=0.3, zorder=0)
    # ax.set_title("")
    ax.set_xlabel('Gaps (days)')
    ax.set_ylabel('CDF')
    ax.legend()

    plt.show()

    return fig, ax

# compare storm per simulation 
def plot_storm_count(res, detected_storm):
    N = [result.size for result in res]

    # plot 
    fig, ax = plt.subplots(figsize=(8,5))
    
    ax.hist(N, bins=50, alpha=0.7, color='tab:blue', edgecolor='black', label='NHPP Thinning Simulation')
    ax.axvline(detected_storm.shape[0], color='red', linestyle='dashed', linewidth=2, label='Observed Data')
    ax.set_xlabel("Number of Storms in 41 years")
    ax.set_ylabel("Frequency")
    ax.set_title("Histogram of Number of Storms in 10,000 NHPP Thinning Simulations")
    ax.text(0.02, 0.87, f'median: {np.median(N)}\nmean: {np.mean(N)}\ndata: {detected_storm.shape[0]}', 
            transform=ax.transAxes, 
            )
    ax.legend()

    plt.show()

    return fig, ax

# compare monthly bar chart of storm count 
def avg_bar_chart(res, date_start, detected_storm):
    
    numsim = len(res)

    # get the average number of storms per month from the simulations
    count_sim = pd.DataFrame(
        {'count': np.zeros(12)},
        index=np.arange(1,13),
    )

    for i in range(numsim):
        out_sl = pd.DataFrame({
            'time':[date_start + np.timedelta64(int(start*24), 'h') for start in res[i]], 
        })

        out_sl['month'] = [s_dt.month for s_dt in out_sl['time']]

        count_sim['addition'] = out_sl.groupby('month').count()['time']
        count_sim['addition'] = count_sim['addition'].fillna(0)
        count_sim['count'] = count_sim['count'] + count_sim['addition']

    count_sim['avg_count'] = count_sim['count'] / numsim

    # get the count for detected storm (data)
    storm_start_data = pd.DataFrame(
        {
            'date_start': helper.datenum_to_datetime(detected_storm['start'])
        }
    )

    count_sim['data_count'] = storm_start_data.groupby(by=storm_start_data['date_start'].dt.month).count()

    # plot the monthly counts 
    fig, ax = plt.subplots(figsize=(10, 5))
    width = 0.4
    x = np.arange(1, 13)

    ax.grid(linestyle='--', alpha=0.3, zorder=0)
    ax.bar(x-width/2, count_sim['avg_count'], width, label= "10,000 Simulation")
    ax.bar(x+width/2, count_sim['data_count'], width, label= "Data")
    ax.set_xticks(x)
    ax.set_xticklabels(['Jan', 'Feb', 'Mar', 'Apr', 'May', 'Jun', 'Jul', 'Aug', 'Sep', 'Oct', 'Nov', 'Dec'])
    ax.set_xlabel("Month")
    ax.set_ylabel("Number of Storms")
    ax.set_title("Monthly Storm Counts: 10,000 HNPP Thinning Simulation vs Data")
    ax.legend()

    plt.show()

    return fig, ax


## Station P2

In [ ]:
# access from the pre-downloaded 
station = 'p2'
data_path = f'../data/ERA5/ts/unzipped/{station}_ts.nc'
ds_p2 = xr.open_dataset(data_path)

### One Realisation

In [ ]:
storm_start_p2, storm_sample_p2, detected_storm_p2 = go_simulate(ds_p2)

dt_storm_start_p2 = date_start + (storm_start_p2 * 24).astype('timedelta64[h]')
dt_detected_storm_start_p2 = helper.datenum_to_datetime(detected_storm_p2['start'])

compare_bar_month(dt_storm_start_p2, dt_detected_storm_start_p2, 'Simulation', 'Data')

In [ ]:
compare_ecdf(storm_start_p2, storm_sample_p2, detected_storm_p2)

### Multiple Realisation

In [ ]:
res_2, durs_pair, detected_storm_2 = go_simulate_multiple(ds_p2, 10000)

In [ ]:
fig, ax = plot_storm_count(res_2, detected_storm_2)

In [ ]:
date_start = ds.valid_time[0].values.astype('datetime64[s]')

fig, ax = avg_bar_chart(res_2, date_start, detected_storm_2)

In [ ]:
fig, ax = compare_ecdf(res_2, durs_pair, detected_storm_p2)

## Station P11

In [ ]:
# access from the pre-downloaded 
station = 'p11'
data_path = f'../data/ERA5/ts/unzipped/{station}_ts.nc'
ds_p11 = xr.open_dataset(data_path)

### Multiple Realisation

In [ ]:
res_p11, dur_pairs_p11, detected_storm_p11 = go_simulate_multiple(ds_p11, 10000)

In [ ]:
fig, ax = plot_storm_count(res_p11, detected_storm_p11)

In [ ]:
date_start = ds.valid_time[0].values.astype('datetime64[s]')

fig, ax = avg_bar_chart(res_p11, date_start, detected_storm_p11)

In [ ]:
fig, ax = compare_ecdf(res_p11, dur_pairs_p11, detected_storm_p11)

## Station P24

In [ ]:
# access from the pre-downloaded 
station = 'p24'
data_path = f'../data/ERA5/ts/unzipped/{station}_ts.nc'
ds_p24 = xr.open_dataset(data_path)

### One Realisation

In [ ]:
storm_start, storm_sample_, detected_storm_ = go_simulate(ds_p24)

In [ ]:
dt_storm_start = date_start + (storm_start * 24).astype('timedelta64[h]')
dt_detected_storm_start = helper.datenum_to_datetime(detected_storm_['start'])

fig, ax = compare_bar_month(dt_storm_start, dt_detected_storm_start, 'Simulation', 'Data')

In [ ]:
fig, ax = compare_ecdf(storm_start, storm_sample_, detected_storm_)

### Multiple Realisation

In [ ]:
res_p24, dur_pairs_p24, detected_storm_p24 = go_simulate_multiple(ds_p24, 10000)

In [ ]:
fig, ax = plot_storm_count(res_p24, detected_storm_p24)

In [ ]:
date_start = ds.valid_time[0].values.astype('datetime64[s]')

fig, ax = avg_bar_chart(res_p24, date_start, detected_storm_p24)

In [ ]:
fig, ax = compare_ecdf(res_p24, dur_pairs_p24, detected_storm_p24)

## SL

In [ ]:
# access from the pre-downloaded 
data_path = '../data/ERA5/ts/sri_lanka_1979_2020.nc'
ds_sl = xr.open_dataset(data_path)

In [ ]:
res_sl, durs_pair_sl, detected_storm_sl = go_simulate_multiple(ds_sl, 10000)

In [ ]:
fig, ax = plot_storm_count(res_sl, detected_storm_sl)

In [ ]:
date_start = ds.valid_time[0].values.astype('datetime64[s]')

fig, ax = avg_bar_chart(res_sl, date_start, detected_storm_sl)

In [ ]:
fig, ax = compare_ecdf(res_sl, durs_pair_sl, detected_storm_sl)

In [ ]:
import importlib
importlib.reload(storm)
importlib.reload(helper)


# Check Kaplan-Meier Scale

Misalkan coba bandingin Station P2 

In [ ]:
def plot_qq(res, detected_storm):

    # search for data quantile 
    times_data = np.asarray((detected_storm['start'].values - 722816), dtype=float)
    n = len(times_data)

    p_data = (np.arange(1, n + 1) - 0.5) / n

    # calculate simulated time quantile 
    times_sim = np.asarray(np.concatenate(res), dtype=float)
    times_sim = np.sort(times_sim)
    n_sim = len(times_sim)

    p_sim = (np.arange(1, n_sim + 1) - 0.5) / n_sim

    # interpolate to data time 
    p_plot_sim = np.interp(times_data, times_sim, p_sim)

    fig, ax = plt.subplots(figsize=(8,6))

    ax.plot(p_plot_sim, p_data, "o", label="KM–Q–Q points")

    lims = [
        min(p_plot_sim.min(), p_plot_sim.min()),
        max(p_data.max(), p_data.max())
    ]

    ax.plot(lims, lims, "k--", label="Perfect agreement")

    ax.set_xlabel("Simulation")
    ax.set_ylabel("ERA 5 Data")
    ax.set_title("KM–Q–Q Plot: Simulation vs Data")

    ax.legend()
    ax.grid(True)
    
    plt.show()

    return fig, ax

In [ ]:
fig, ax = plot_qq(res_2, detected_storm_p2)

In [ ]:
fig, ax = plot_qq(res_p11, detected_storm_p11)

In [ ]:
fig, ax = plot_qq(res_p24, detected_storm_p24)


In [ ]:
fig, ax = plot_qq(res_sl, detected_storm_sl)


# 